# 04 — Entrenamiento con augmentation (Fase 3)

**Objetivo:** mejorar el baseline de Fase 2 probando técnicas de augmentation.

Plan de Fase 3 (orden de pruebas):
1. **Mixup sobre embeddings** (esta corrida) — barato, no requiere re-extraer embeddings.
2. *Audio augmentation* (audiomentations sobre waveform → re-extraer embeddings → `embeddings:v1`) — pendiente.
3. *Combinación* (audio aug + mixup) — pendiente.

**Target:** macro-F1 ≥ 0.80 en `test_hard`.

**Logging:** todas las corridas van a W&B con tag `aug` o `mixup`. Comparar contra `baseline-v0` en la UI de W&B.

## 1. Setup del entorno Colab

Clonamos el repo desde GitHub e instalamos dependencias.

> Si ya clonaste antes y querés actualizar, corré la celda igual — hace `git pull` si la carpeta ya existe.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Placaflaca00/bird_classifierPY.git"
REPO_DIR = Path("/content/birdClassifier")

if REPO_DIR.exists():
    print("Repo ya existe, haciendo git pull...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    print("Clonando repo...")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(f"\ncwd: {os.getcwd()}")

In [ ]:
# Instalar el package en modo editable con extras de data pipeline.
# Tarda ~2-3 min la primera vez (descarga torch/lightning/pandas/etc).
!pip install -q -e ".[data]"

## 2. Credenciales W&B

Leemos `WANDB_API_KEY` y `WANDB_ENTITY` desde **Colab Secrets** (ícono 🔑 en la sidebar izquierda → "Add new secret").

Secretos que tenés que crear una sola vez:
- `WANDB_API_KEY` → tu key de https://wandb.ai/authorize
- `WANDB_ENTITY`  → tu username/team de W&B

El `WANDB_PROJECT` es público (`bird-classifier-py`), va hardcoded.

> **Nunca pegues la API key en una celda.** Si commiteás el notebook con la key adentro, queda en el historial de git para siempre.

In [ ]:
from google.colab import userdata

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["WANDB_ENTITY"] = userdata.get("WANDB_ENTITY")
os.environ["WANDB_PROJECT"] = "bird-classifier-py"

print("WANDB_PROJECT:", os.environ["WANDB_PROJECT"])
print("WANDB_ENTITY:", os.environ["WANDB_ENTITY"])
print("WANDB_API_KEY:", "***" + os.environ["WANDB_API_KEY"][-4:])  # solo últimos 4 chars

## 3. Run: mixup_alpha = 0.2

Primera prueba: mixup conservador (`α=0.2`, recomendado por el paper original de Zhang et al. 2018 para datasets chicos).

Reusa **`embeddings:v0`** (los mismos del baseline) — el cambio es 100% en el training loop, no se re-extrae nada.

Hyperparams: idénticos al baseline excepto `mixup_alpha`. Así el delta en macro-F1 es atribuible a mixup y no a otra cosa.

In [ ]:
from src.training.train import train

result_mixup_02 = train({
    "mixup_alpha": 0.2,
    "run_name": "baseline-v0-mixup-0.2",
    "tags": ["phase3", "mixup", "alpha-0.2"],
})

print("\n=== Resumen ===")
print(f"Best val_macro_f1: {result_mixup_02['best_val_macro_f1']:.4f}")
for fold, metrics in result_mixup_02["final_results"].items():
    print(f"  {fold:<11} macro_f1={metrics['test_macro_f1']:.4f}  acc={metrics['test_acc']:.4f}")

## 4. Comparación con baseline

Ir a https://wandb.ai/<tu-entity>/bird-classifier-py y comparar las dos runs lado a lado:
- `baseline-v0` (sin mixup)
- `baseline-v0-mixup-0.2` (esta corrida)

**Métricas a mirar:**
- `final/test_hard_macro_f1` — la métrica primaria (target ≥ 0.80).
- `final/val_macro_f1` — sanity check (no debería bajar).
- Curvas de `train_loss` vs `val_loss` — con mixup el train loss típicamente sube (porque el modelo ve ejemplos "más difíciles") pero val loss debería bajar o quedar igual. Si val loss sube → mixup está hiriendo, no ayudando.

**Próximos pasos** (pendientes según resultado):
- Si mejora: probar `mixup_alpha ∈ {0.4, 1.0}` para ver si conviene más fuerte.
- Si no mejora: pasar directo a Pieza 2 (audio augmentation con audiomentations).